# Load Data

In [59]:
# USE THIS WHEN RUNNING ON COLAB

"""
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/combined_ipl_mbm_bowl.csv')"""

'\nimport kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf = pd.read_csv(path + \'/combined_ipl_mbm_bowl.csv\')'

In [5]:
# USE THIS WHEN RUNNING LOCALLY

import pandas as pd
import numpy as np

df = pd.read_csv(r"../../data/mbm/all_league_bowl_final.csv")

In [6]:
df['date'] = pd.to_datetime(df['date'])

df.sort_values(by=['date', 'inns'], inplace=True)

df['season'] = df['date'].dt.year

df.drop(columns = ['maidens', 'eco', 'lbw_or_bowled', 'dots'], inplace=True)

# Load player details
players_detail = pd.read_csv(r"../../data/register_data/people3.csv")

# Create a clean mapping dictionary by filtering out NaN values
valid_players = players_detail.dropna(subset=['key_cricinfo'])
name_corrector = dict(zip(valid_players['key_cricinfo'], valid_players['unique_name']))

# Apply the mapping without any fallback - missing values will remain as NaN
df['bowler'] = df['bowler_id'].map(name_corrector)

# Career Stats

In [7]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Compute cumulative statistics
df['cumulative_runs'] = df.groupby(['bowler_id', 'league'])['runs'].cumsum() - df['runs']
df['cumulative_wickets'] = df.groupby(['bowler_id', 'league'])['wickets'].cumsum() - df['wickets']
df['cumulative_balls'] = df.groupby(['bowler_id', 'league'])['total_balls'].cumsum() - df['total_balls']
df['cum_matches'] = df.groupby(['bowler_id', 'league']).cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute career bowling metrics
df['career_avg'] = df['cumulative_runs'] / df['cumulative_wickets']
df['career_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6).replace(0, 1)
df['career_sr'] = df['cumulative_balls'] / df['cumulative_wickets']
df['career_wpm'] = df['cumulative_wickets'] / df['cum_matches']
df['career_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs', 'cumulative_wickets', 'cumulative_balls', 'cum_matches', 'cumulative_overs'], inplace=True)

# Columns to process
career_cols = ['career_avg', 'career_eco', 'career_sr']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in career_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'league']).head(1).index, career_cols] = pd.NA

In [8]:
df[df['bowler_id'] == 625383].head()

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,wickets,balls,venue,league,season,career_avg,career_eco,career_sr,career_wpm,career_opm
15921,15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,3,24,M.Chinnaswamy Stadium,IPL,2013,NaN,NaN,NaN,NaN,NaN
16481,16481,598024,2013-04-21,2,JJ Bumrah,625383,Mumbai Indians,Delhi Capitals,3.0,38.0,0,21,Feroz Shah Kotla,IPL,2013,10.666667,8.000000,8.0,3.00,4.00
24739,24739,729287,2014-04-19,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,2.0,14.0,0,12,Dubai International Cricket Stadium,IPL,2014,23.333333,10.000000,14.0,1.50,3.50
25155,25155,733973,2014-05-03,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,25.0,0,27,Wankhede Stadium,IPL,2014,28.000000,9.333333,18.0,1.00,3.00
25276,25276,733983,2014-05-06,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,22.0,2,24,Wankhede Stadium,IPL,2014,36.333333,8.384615,26.0,0.75,3.25


In [9]:
df[df['bowler_id'] == 625383].info() # Career columns can have one row of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 15921 to 160957
Data columns (total 20 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  210 non-null    int64         
 1   match_id    210 non-null    int64         
 2   date        210 non-null    datetime64[ns]
 3   inns        210 non-null    int64         
 4   bowler      210 non-null    object        
 5   bowler_id   210 non-null    int64         
 6   bowl_team   210 non-null    object        
 7   bat_team    210 non-null    object        
 8   overs       210 non-null    object        
 9   runs        210 non-null    float64       
 10  wickets     210 non-null    int64         
 11  balls       210 non-null    int64         
 12  venue       210 non-null    object        
 13  league      210 non-null    object        
 14  season      210 non-null    int32         
 15  career_avg  207 non-null    float64       
 16  career_eco  207 non-null

In [10]:
df[(df['bowler_id'] == 625383) & (df['career_avg'].isna())]

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,wickets,balls,venue,league,season,career_avg,career_eco,career_sr,career_wpm,career_opm
15921,15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,3,24,M.Chinnaswamy Stadium,IPL,2013,NaN,NaN,NaN,NaN,NaN
34847,34847,902087,2016-01-02,2,JJ Bumrah,625383,Gujarat,Himachal Pradesh,4.0,24.0,1,25,Vidarbha Cricket Association Stadium,SMA,2016,NaN,NaN,NaN,NaN,NaN
34077,34077,895817,2016-01-26,2,JJ Bumrah,625383,India,Australia,3.3,23.0,3,22,Adelaide Oval,T20IS,2016,NaN,NaN,NaN,NaN,NaN


# Last 5 match stats

In [11]:
# Sort bowling data by bowler and match date
df.sort_values(by=['bowler', 'date'], inplace=True)

# ------------------------------------------------------------------------------
# 1) Bowler Last 5 Match Rolling Bowler's Average
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('bowler_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_wickets = df.groupby('bowler_id')['wickets'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
# Only calculate average when there are wickets taken
df['last5_avg'] = rolling_runs / rolling_wickets
# Replace infinite values with NaN
df['last5_avg'] = df['last5_avg'].replace([np.inf, -np.inf], 0)
df['last5_avg'] = df['last5_avg'].fillna(0)

# ------------------------------------------------------------------------------
# 2) Bowler Last 5 Match Rolling Economy Rate
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('bowler_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_overs = df.groupby('bowler_id')['overs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
df['last5_eco'] = rolling_runs / rolling_overs.replace(0, 1)

# ------------------------------------------------------------------------------
# 3) Bowler Last 5 Match Rolling Strike Rate (balls per wicket)
# ------------------------------------------------------------------------------
rolling_balls = df.groupby('bowler_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
# Only calculate strike rate when there are wickets taken
df['last5_sr'] = rolling_balls / rolling_wickets
# Replace infinite values with NaN
df['last5_sr'] = df['last5_sr'].replace([np.inf, -np.inf], 0)
df['last5_sr'] = df['last5_sr'].fillna(0)

# ------------------------------------------------------------------------------
# 4) Bowler Last 5 Match Rolling Wickets per Match
# ------------------------------------------------------------------------------
df['last5_wpm'] = df.groupby('bowler_id')['wickets'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 5) Bowler Last 5 Match Rolling Overs per Match
# ------------------------------------------------------------------------------
# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']


df['last5_opm'] = df.groupby('bowler_id')['total_balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()/6
)

df.drop(columns=['overs_int', 'balls_int', 'total_balls'], inplace=True)

last5_cols = ['last5_avg', 'last5_eco', 'last5_sr']
# making the first val as Null
df.loc[df.groupby(['bowler_id']).head(1).index, last5_cols] = pd.NA

In [12]:
df[df['bowler_id'] == 625383].head()

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,...,career_avg,career_eco,career_sr,career_wpm,career_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm
15921,15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16481,16481,598024,2013-04-21,2,JJ Bumrah,625383,Mumbai Indians,Delhi Capitals,3.0,38.0,...,10.666667,8.000000,8.0,3.00,4.00,10.666667,8.000000,8.0,3.00,4.00
24739,24739,729287,2014-04-19,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,2.0,14.0,...,23.333333,10.000000,14.0,1.50,3.50,23.333333,10.000000,15.0,1.50,3.50
25155,25155,733973,2014-05-03,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,25.0,...,28.000000,9.333333,18.0,1.00,3.00,28.000000,9.333333,19.0,1.00,3.00
25276,25276,733983,2014-05-06,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,22.0,...,36.333333,8.384615,26.0,0.75,3.25,36.333333,8.384615,28.0,0.75,3.25


In [13]:
df[df['bowler_id'] == 625383].info() # Last 5 match columns can have one row of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 15921 to 160957
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  210 non-null    int64         
 1   match_id    210 non-null    int64         
 2   date        210 non-null    datetime64[ns]
 3   inns        210 non-null    int64         
 4   bowler      210 non-null    object        
 5   bowler_id   210 non-null    int64         
 6   bowl_team   210 non-null    object        
 7   bat_team    210 non-null    object        
 8   overs       210 non-null    object        
 9   runs        210 non-null    float64       
 10  wickets     210 non-null    int64         
 11  balls       210 non-null    int64         
 12  venue       210 non-null    object        
 13  league      210 non-null    object        
 14  season      210 non-null    int32         
 15  career_avg  207 non-null    float64       
 16  career_eco  207 non-null

# Venue stats

In [14]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and venue to compute cumulative statistics
grouped = df.groupby(['bowler_id', 'venue'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute venue-wise bowling metrics
df['venue_avg'] = df['cumulative_runs'] / df['cumulative_wickets']
df['venue_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6)
df['venue_sr'] = df['cumulative_balls'] / df['cumulative_wickets']
df['venue_wpm'] = df['cumulative_wickets'] / df['cum_matches']
df['venue_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
venue_cols = ['venue_avg', 'venue_eco', 'venue_sr']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in venue_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'venue']).head(1).index, venue_cols] = pd.NA

In [15]:
df[(df['bowler_id'] == 625383) & (df['venue'] == 'Wankhede Stadium')].head()

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,...,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm,venue_avg,venue_eco,venue_sr,venue_wpm,venue_opm
25155,25155,733973,2014-05-03,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,25.0,...,28.000000,9.333333,19.0,1.00,3.00,NaN,NaN,NaN,NaN,NaN
25276,25276,733983,2014-05-06,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,22.0,...,36.333333,8.384615,28.0,0.75,3.25,0.0,6.250000,0.0,0.000000,4.0
25409,25409,733995,2014-05-10,2,JJ Bumrah,625383,Mumbai Indians,Chennai Super Kings,4.0,33.0,...,26.200000,7.705882,21.6,1.00,3.40,23.5,5.875000,24.0,1.000000,4.0
25803,25803,734031,2014-05-23,2,JJ Bumrah,625383,Mumbai Indians,Delhi Capitals,4.0,36.0,...,67.000000,7.444444,55.5,0.40,3.60,40.0,6.666667,36.0,0.666667,4.0
25903,25903,734041,2014-05-25,1,JJ Bumrah,625383,Mumbai Indians,Rajasthan Royals,4.0,30.0,...,68.500000,7.611111,55.0,0.40,3.60,58.0,7.250000,48.0,0.500000,4.0


In [16]:
print("No of unique venues played at by Jasprit Bumrah : ",len(df[(df['bowler_id'] == 625383)]['venue'].unique()))
df[(df['bowler_id'] == 625383)].info() # NOTE - Venue stats can have as many row of null values for each bowler as the number of venues played at

No of unique venues played at by Jasprit Bumrah :  53
<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 15921 to 160957
Data columns (total 30 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  210 non-null    int64         
 1   match_id    210 non-null    int64         
 2   date        210 non-null    datetime64[ns]
 3   inns        210 non-null    int64         
 4   bowler      210 non-null    object        
 5   bowler_id   210 non-null    int64         
 6   bowl_team   210 non-null    object        
 7   bat_team    210 non-null    object        
 8   overs       210 non-null    object        
 9   runs        210 non-null    float64       
 10  wickets     210 non-null    int64         
 11  balls       210 non-null    int64         
 12  venue       210 non-null    object        
 13  league      210 non-null    object        
 14  season      210 non-null    int32         
 15  career_avg  207 no

# Innings Stats

In [17]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and innings
grouped = df.groupby(['bowler_id', 'inns','league'])

# Compute cumulative statistics for each bowler in each innings
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']  # Subtract current row's runs
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']  # Subtract current row's wickets
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']  # Subtract current row's balls
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute innings-wise bowling metrics
df['inns_avg'] = df['cumulative_runs'] / df['cumulative_wickets']  # Average runs per wicket
df['inns_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6).replace(0,1)  # Economy rate
df['inns_sr'] = df['cumulative_balls'] / df['cumulative_wickets']  # Strike rate (balls per wicket)
df['inns_wpm'] = df['cumulative_wickets'] / df['cum_matches']  # Wickets per match
df['inns_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']  # Overs per match

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
inns_cols = ['inns_avg', 'inns_sr', 'inns_eco']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in inns_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'inns','league']).head(1).index, inns_cols] = pd.NA

In [18]:
df[(df['bowler_id'] == 625383) & (df['inns'] == 1)].head()

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,...,venue_avg,venue_eco,venue_sr,venue_wpm,venue_opm,inns_avg,inns_eco,inns_sr,inns_wpm,inns_opm
15921,15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25155,25155,733973,2014-05-03,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,25.0,...,NaN,NaN,NaN,NaN,NaN,10.666667,8.000,8.0,3.00,4.0
25463,25463,734001,2014-05-12,1,JJ Bumrah,625383,Mumbai Indians,Sunrisers Hyderabad,4.0,24.0,...,NaN,NaN,NaN,NaN,NaN,19.000000,7.125,16.0,1.50,4.0
25728,25728,734025,2014-05-21,1,JJ Bumrah,625383,Mumbai Indians,Punjab Kings,4.0,31.0,...,NaN,NaN,NaN,NaN,NaN,27.000000,6.750,24.0,1.00,4.0
25903,25903,734041,2014-05-25,1,JJ Bumrah,625383,Mumbai Indians,Rajasthan Royals,4.0,30.0,...,58.0,7.25,48.0,0.5,4.0,22.400000,7.000,19.2,1.25,4.0


In [19]:
df[(df['bowler_id'] == 625383)].info() # Innings stats can have two rows of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 15921 to 160957
Data columns (total 35 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  210 non-null    int64         
 1   match_id    210 non-null    int64         
 2   date        210 non-null    datetime64[ns]
 3   inns        210 non-null    int64         
 4   bowler      210 non-null    object        
 5   bowler_id   210 non-null    int64         
 6   bowl_team   210 non-null    object        
 7   bat_team    210 non-null    object        
 8   overs       210 non-null    object        
 9   runs        210 non-null    float64       
 10  wickets     210 non-null    int64         
 11  balls       210 non-null    int64         
 12  venue       210 non-null    object        
 13  league      210 non-null    object        
 14  season      210 non-null    int32         
 15  career_avg  207 non-null    float64       
 16  career_eco  207 non-null

In [20]:
df[(df['bowler_id'] == 625383) & (df['inns_avg'].isnull())]

,Unnamed: 0,match_id,date,inns,bowler,bowler_id,bowl_team,bat_team,overs,runs,...,venue_avg,venue_eco,venue_sr,venue_wpm,venue_opm,inns_avg,inns_eco,inns_sr,inns_wpm,inns_opm
15921,15921,597999,2013-04-04,1,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,4.0,32.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16481,16481,598024,2013-04-21,2,JJ Bumrah,625383,Mumbai Indians,Delhi Capitals,3.0,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34847,34847,902087,2016-01-02,2,JJ Bumrah,625383,Gujarat,Himachal Pradesh,4.0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35366,35366,902137,2016-01-04,1,JJ Bumrah,625383,Gujarat,Hyderabad (India),4.0,30.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34077,34077,895817,2016-01-26,2,JJ Bumrah,625383,India,Australia,3.3,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34109,34109,895821,2016-01-31,1,JJ Bumrah,625383,India,Australia,4.0,43.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Season Stats

In [21]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and season
grouped = df.groupby(['bowler_id', 'season','league'])

# Compute cumulative statistics for each bowler in each season
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']  # Subtract current row's runs
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']  # Subtract current row's wickets
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']  # Subtract current row's balls
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute season-wise bowling metrics
df['ssn_avg'] = df['cumulative_runs'] / df['cumulative_wickets'] # Average runs per wicket
df['ssn_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6)  # Economy rate
df['ssn_sr'] = df['cumulative_balls'] / df['cumulative_wickets']  # Strike rate (balls per wicket)
df['ssn_wpm'] = df['cumulative_wickets'] / df['cum_matches']  # Wickets per match
df['ssn_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']  # Overs per match

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
ssn_cols = ['ssn_avg', 'ssn_sr', 'ssn_eco']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in ssn_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'season','league']).head(1).index, ssn_cols] = pd.NA

In [23]:
print("No of unique seasons played by Jasprit Bumrah : ",len(df[(df['bowler_id'] == 625383)]['season'].unique()))
df[(df['bowler_id'] == 625383)].info() # NOTE - Season stats can have as many row of null values for each bowler as the number of seasons played

No of unique seasons played by Jasprit Bumrah :  13
<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 15921 to 160957
Data columns (total 40 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  210 non-null    int64         
 1   match_id    210 non-null    int64         
 2   date        210 non-null    datetime64[ns]
 3   inns        210 non-null    int64         
 4   bowler      210 non-null    object        
 5   bowler_id   210 non-null    int64         
 6   bowl_team   210 non-null    object        
 7   bat_team    210 non-null    object        
 8   overs       210 non-null    object        
 9   runs        210 non-null    float64       
 10  wickets     210 non-null    int64         
 11  balls       210 non-null    int64         
 12  venue       210 non-null    object        
 13  league      210 non-null    object        
 14  season      210 non-null    int32         
 15  career_avg  207 non-

# Cleaning and Exporting

In [24]:
df.drop(columns=['overs', 'runs', 'wickets', 'balls'], inplace=True)

In [25]:
df = df.reindex(columns=['date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler', 'inns', 'venue', 'season','league',
                         'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm',
                         'inns_avg', 'inns_eco', 'inns_sr', 'inns_wpm', 'inns_opm',
                         'venue_avg', 'venue_eco', 'venue_sr', 'venue_wpm', 'venue_opm',
                         'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm', 'ssn_opm',
                         'last5_avg', 'last5_eco', 'last5_sr', 'last5_wpm', 'last5_opm'])

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 161041 entries, 66495 to 7415
Data columns (total 35 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        161041 non-null  datetime64[ns]
 1   match_id    161041 non-null  int64         
 2   bowl_team   161041 non-null  object        
 3   bat_team    161041 non-null  object        
 4   bowler_id   161041 non-null  int64         
 5   bowler      161041 non-null  object        
 6   inns        161041 non-null  int64         
 7   venue       161041 non-null  object        
 8   season      161041 non-null  int32         
 9   league      161041 non-null  object        
 10  career_avg  152669 non-null  float64       
 11  career_eco  152669 non-null  float64       
 12  career_sr   152669 non-null  float64       
 13  career_wpm  152669 non-null  float64       
 14  career_opm  152669 non-null  float64       
 15  inns_avg    145701 non-null  float64       
 16  inns_

In [27]:
df['date'].min(), df['date'].max()

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-04-07 00:00:00'))

In [28]:
df.to_csv(r"D:\ED\FIFS_IPLungit\data\processed_data\all_league_bowl_features.csv", index=False)